In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.9 Tight Binding: From Chain to Graphene

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.9",
    title="Tight Binding: From Chain to Graphene",
    blurb="Movement III moves the many-electron machinery into crystals, starting "
    "from the chemist's limit: electrons hopping between atomic orbitals. One "
    "matrix element generates the band structures of a chain, a square lattice, "
    "and graphene's honeycomb — whose two famous Dirac points emerge, exactly, "
    "from three complex numbers summing to zero. Densities of states acquire "
    "their dimensional fingerprints (inverse-square-root edges, a logarithmic "
    "saddle matching its analytic coefficient to a percent), the "
    "Hellmann–Feynman theorem starts computing forces, and a Peierls "
    "dimerization closes the loop from bands back to structure.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

Movements I and II built the machinery of interacting electrons in atoms;
Movement III puts electrons in *crystals*, where
[§7.12](../07-quantum-statistical-mechanics/bloch-theorem-band-structure.ipynb)
established the grammar (Bloch's theorem, bands, the inert filled band) and this
volume now supplies the working vocabulary. Tight binding is the chemist's
entrance: start from atomic orbitals, let electrons hop to neighbors with
amplitude $-t$, and diagonalize in the Bloch basis. The method is almost
embarrassingly simple and almost embarrassingly powerful — it undergirds
everything from back-of-envelope band estimates to the Wannier-function
machinery of [§8.12](berry-wannier-ssh.ipynb) and the Hubbard models of
[§8.13](hubbard-model.ipynb), and in 2004 it turned out to describe, essentially
exactly, the most celebrated material of the century.

The build: the monatomic chain from the LCAO secular equation (with the
usually-dropped overlap correction computed rather than waved off); the square
lattice and the dimensional gallery of densities of states — inverse-square-root
edges in one dimension, a logarithmic van Hove saddle in two whose coefficient
matches its analytic value $1/2\pi^2 t$ to better than a percent; then graphene,
where the honeycomb's two-site unit cell gives a $2\times2$ Bloch Hamiltonian
whose off-diagonal element vanishes *exactly* at the Brillouin-zone corners:
Dirac cones, a measured Fermi velocity of precisely $3ta/2$, and a linear,
vanishing density of states. The closing exercises put the bands to work:
Hellmann–Feynman forces certified against finite differences at $10^{-6}$, and
the Peierls instability — a half-filled chain *lowering its electronic energy by
dimerizing* — computed as the volume's first band-structure-drives-structure
result.

> **Conventions (this notebook).** Hopping $t = 1$ sets the energy unit; bond
> lengths set length units (chain spacing $a = 1$; graphene carbon–carbon
> distance $a = 1$, so the lattice vectors are
> $\mathbf a_{1,2} = (3/2, \pm\sqrt3/2)$). On-site energies are zero unless
> stated. Bloch Hamiltonians are explicit `numpy` arrays diagonalized by
> `numpy.linalg.eigh` (or read off analytically where $2\times2$); densities of
> states are normalized histograms (`numpy.histogram`, `density=True`) over
> uniformly sampled Brillouin zones.
>
> **How to read the checks.** Each exercise closes with a `validate` call
> against an independent fact: an exact zero at a symmetry point, an analytic
> coefficient, a fitted exponent, a force identity. A ✓ is strong evidence; a ✗
> is a prompt to locate the discrepancy, not an automatic verdict.
>
> **Scope.** Orthogonal single-orbital tight binding plus the first overlap
> correction; multi-orbital fits to ab-initio bands, spin–orbit terms, and
> Slater–Koster tables are surveyed in Ashcroft & Mermin and in Martin
> {cite}`martin2004`, Ch. 14. Graphene's electronic story is told in full in
> the standard reviews; the tight-binding treatment here is the one Wallace
> wrote down in 1947, sixty years before the Nobel prize it described.

## Theory in brief

### The LCAO secular problem, and the band of a chain

Place one orbital $|j\rangle$ on each site of a lattice and expand the crystal
state in them. Bloch's theorem
([§7.12](../07-quantum-statistical-mechanics/bloch-theorem-band-structure.ipynb))
fixes the coefficients to phases, $|k\rangle = \sum_j e^{ikja}|j\rangle$, and
the variational principle in this basis gives the generalized eigenproblem
$\big[\varepsilon_{\mathrm{AT}} - \varepsilon(k)\big] S(k) + A(k) = 0$, with
$A(k)$ the Hamiltonian's Bloch sum and $S(k)$ the overlap's. For a chain with
nearest-neighbor hopping $\langle j|\hat H|j\pm1\rangle = -t$ and overlap
$\langle j|j\pm1\rangle = s$,

```{math}
:label: eq-tb-chain
\varepsilon(k) = \frac{\varepsilon_{\mathrm{AT}} - 2t\cos ka}{1 + 2s\cos ka}
\;\xrightarrow{\;s\to0\;}\;
\varepsilon_{\mathrm{AT}} - 2t\cos ka :
```

the textbook cosine band of width $4t$, with the overlap correction skewing it
asymmetrically (bonding states pushed less than antibonding, because
$S > 1$ where the orbitals add). One orbital per site, one band; $N$ sites, $N$
states; the band *is* the atomic level, delocalized.

### Densities of states and van Hove's theorem

The number of states per energy,
$g(\varepsilon) = \sum_{\mathbf k}\delta(\varepsilon - \varepsilon_{\mathbf k})$,
inherits singularities wherever the band flattens ($\nabla_{\mathbf k}
\varepsilon = 0$), and their *shape* is fixed by dimension alone (van Hove
1953; Ashcroft & Mermin, Ch. 8, classify the cases): in one dimension band
edges diverge as $|\varepsilon - \varepsilon_c|^{-1/2}$, in two dimensions a
saddle point produces a symmetric logarithmic peak, and in three dimensions
only square-root *kinks* survive. For the square lattice,
$\varepsilon = -2t(\cos k_x a + \cos k_y a)$, the saddle at
$(\pi/a, 0)$ gives the closed form

```{math}
:label: eq-tb-vanhove
g(\varepsilon) \;\approx\; \frac{1}{2\pi^2 t}\,
\ln\frac{16t}{|\varepsilon|}
\qquad (|\varepsilon| \ll t),
```

whose coefficient $1/2\pi^2 t = 0.0507$ the histogram below reproduces to
better than a percent — a rare chance to fit a *logarithm's prefactor* against
theory.

### Graphene: two sites, one zero, massless electrons

The honeycomb lattice is triangular with a two-atom basis (A and B
sublattices), so the Bloch Hamiltonian is a $2\times2$ matrix,

```{math}
:label: eq-tb-graphene
H(\mathbf k) = \begin{pmatrix} 0 & -t\,f(\mathbf k) \\
-t\,f^*(\mathbf k) & 0 \end{pmatrix},
\qquad
f(\mathbf k) = 1 + e^{i\mathbf k\cdot\mathbf a_1} + e^{i\mathbf k\cdot\mathbf a_2},
\qquad
\varepsilon_\pm = \pm\, t\,|f(\mathbf k)| ,
```

with $f$ the sum of phases to the three B neighbors of an A atom. The entire
electronic identity of graphene hangs on one question: can three unit phasors
sum to zero? At the Brillouin-zone corner $\mathbf K$ the three phases are
$1, e^{2\pi i/3}, e^{4\pi i/3}$ — the cube roots of unity — and their sum
vanishes *identically*. The two bands touch at isolated points, disperse
linearly around them with slope $v_F = 3ta/2$ (measured below by fit), and the
density of states vanishes linearly at the touching: electrons that behave, in
every kinematic respect, like massless Dirac particles with $c \to v_F$. The
half-filled honeycomb sits its Fermi level exactly on the touching points:
graphene is a semimetal by counting alone.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

T_HOP = 1.0  # the hopping integral: the energy unit throughout

# graphene geometry (bond length 1): lattice vectors and the K corner
A1 = np.array([1.5, np.sqrt(3.0) / 2.0])
A2 = np.array([1.5, -np.sqrt(3.0) / 2.0])
K_POINT = np.array([2.0 * np.pi / 3.0, 2.0 * np.pi / (3.0 * np.sqrt(3.0))])


def f_graphene(kx, ky):
    """The graphene Bloch factor f(k) = 1 + exp(i k.a1) + exp(i k.a2).

    The sum of phases from an A atom to its three B neighbors; the band
    energies are +- t |f|, and every Dirac miracle is a zero of this one
    complex function.

    Parameters
    ----------
    kx, ky : float or numpy.ndarray
        Wavevector components (bond length = 1 units).

    Returns
    -------
    complex or numpy.ndarray
        The complex Bloch factor.
    """
    return (
        1.0
        + np.exp(1j * (kx * A1[0] + ky * A1[1]))
        + np.exp(1j * (kx * A2[0] + ky * A2[1]))
    )


def dimer_chain(n_sites, delta):
    """Open dimerized chain Hamiltonian with alternating hoppings t(1 +- delta).

    The Peierls test bench: bond i carries hopping -(1 + delta (-1)^i), so
    delta = 0 is the uniform chain and delta > 0 alternates strong/weak bonds.

    Parameters
    ----------
    n_sites : int
        Number of sites (open boundaries).
    delta : float
        Dimerization amplitude.

    Returns
    -------
    numpy.ndarray
        The (n_sites, n_sites) real symmetric Hamiltonian.
    """
    ham = np.zeros((n_sites, n_sites))
    for i in range(n_sites - 1):
        ham[i, i + 1] = ham[i + 1, i] = -(1.0 + delta * (-1.0) ** i)
    return ham

## Exercise 1 — The chain, with the honest overlap

Equation {eq}`eq-tb-chain` is usually quoted at $s = 0$; the full form is a
two-line computation and teaches something real about bonding.

**Part a)** Plot $\varepsilon(k)$ over the Brillouin zone $k \in [-\pi, \pi]$
for $s = 0$ and $s = 0.1$ (with $\varepsilon_{\mathrm{AT}} = 0$, `numpy`
arithmetic on Eq. {eq}`eq-tb-chain`) and report the band edges. At $s = 0$ the
band is the symmetric $[-2t, 2t]$ cosine; at $s = 0.1$ the *bonding* edge
($k = 0$) sits at $-2t/1.2 = -1.667t$ while the *antibonding* edge ($k = \pi$)
is pushed to $+2t/0.8 = +2.5t$ — overlap widens antibonding more than it
deepens bonding, the same asymmetry every quantum-chemistry course meets in
H$_2^+$.

**Part b)** Verify the two closed-form edges by `numpy` evaluation at
$k = 0, \pi$ and confirm the $s = 0$ bandwidth is exactly $4t$.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the edges, both ways

The $s = 0$ bandwidth must be exactly $4t$, and the $s = 0.1$ edges must land
on their closed forms $-1.6667t$ and $+2.5t$.

In [ ]:
validate.close(
    float(np.ptp(band_s0)), 4.0, "the orthogonal chain's bandwidth 4t", rtol=1e-9
)
validate.close(
    float(band_s1.min()), edge_bond, "the bonding edge with overlap", rtol=1e-9
)
validate.close(
    float(band_s1.max()), edge_anti, "the antibonding edge with overlap", rtol=1e-9
)

## Exercise 2 — The dimensional gallery of densities of states

Van Hove's theorem says dimension writes its signature on $g(\varepsilon)$, and
three histograms make the signatures legible. All three lattices are sampled
uniformly over their Brillouin zones and binned with `numpy.histogram`
(`density=True`).

**Part a)** The chain: sample $\varepsilon = -2t\cos ka$ on $2\times10^5$
$k$-points, and fit the upper band edge's divergence — the slope of
$\ln g$ against $\ln(2t - \varepsilon)$ over $\varepsilon \in [1.5t, 1.99t]$
(`numpy.polyfit`) must come out $-1/2$.

**Part b)** The square lattice: sample
$\varepsilon = -2t(\cos k_x a + \cos k_y a)$ on a $2001^2$ mesh
(`numpy.meshgrid`), and fit the saddle's logarithm: regressing
$g(\varepsilon)$ on $\ln|\varepsilon|$ over $0.05t < |\varepsilon| < t$ must
return the *analytic prefactor* $-1/2\pi^2 t = -0.0507$ of
Eq. {eq}`eq-tb-vanhove` — theory pinned to two digits by a histogram.

**Part c)** Plot the two densities of states with their singularities marked;
graphene's (linear at zero) joins the gallery in Exercise 4.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — dimension's signatures, fitted

The chain's edge exponent must be $-1/2$ within the histogram's resolution,
and the square lattice's logarithmic prefactor must match the analytic
$-1/2\pi^2$ within $2\%$.

In [ ]:
validate.close(exp_1d, -0.5, "the 1-D inverse-square-root edge", rtol=8e-2)
validate.close(
    log_coeff,
    -1.0 / (2.0 * np.pi**2),
    "the square saddle's analytic prefactor",
    rtol=2e-2,
)

## Exercise 3 — Graphene's bands, and the exact zero

Equation {eq}`eq-tb-graphene` reduces graphene to one complex function, and its
claimed zero at the zone corner is exact arithmetic: at $\mathbf K$ the three
phases are the cube roots of unity.

**Part a)** Evaluate the Setup helper `f_graphene` at $\mathbf K =
(2\pi/3,\, 2\pi/3\sqrt3)$: $|f(\mathbf K)|$ must vanish at machine precision —
a band gap of exactly zero, protected by geometry rather than tuning.

**Part b)** Plot the bands $\pm t|f|$ along the standard path
$\Gamma \to K \to M \to \Gamma$ ($M = (2\pi/3, 0)$; `numpy.linspace` segments)
and, as an inset, the lower band's surface near $\mathbf K$ (a `numpy.meshgrid`
patch): the cone. Report the band extremes at $\Gamma$ ($\pm3t$: all three
phasors aligned) and $M$ ($\pm t$).

In [ ]:
# (solution hidden on the public site)


### Validation 3 — geometry, not tuning

$|f(\mathbf K)|$ must vanish at machine precision, and the path extremes must
be exactly 3 (at $\Gamma$) and 1 (at $M$).

In [ ]:
validate.close(
    f_at_K, 0.0, "the Dirac point: three phasors cancel exactly", rtol=0.0, atol=1e-12
)
validate.close(eps_gamma, 3.0, "the Gamma-point energy 3t", rtol=1e-12)
validate.close(eps_m, 1.0, "the M-point energy t", rtol=1e-12)

## Exercise 4 — Massless: the Fermi velocity and the linear DOS

Around its zeros $f$ is linear in the displacement, so the bands form cones
$\varepsilon = \pm v_F|\mathbf q|$, and expanding Eq. {eq}`eq-tb-graphene`
gives $v_F = 3ta/2$ — with $a$ the carbon–carbon bond, the one length in the
problem. Both the slope and its consequence for the density of states are
measurable.

**Part a)** Fit the cone: evaluate $t|f|$ at displacements
$q \in [10^{-4}, 10^{-2}]$ from $\mathbf K$ along $k_x$
(`numpy.geomspace`), fit with degree-1 `numpy.polyfit`, and compare the slope
with $3ta/2 = 1.5$.

**Part b)** The graphene density of states: histogram $\pm t|f|$ over a dense
Brillouin-zone sampling ($1200^2$ mesh). Verify the three signatures: $g$
vanishes at zero energy (the Dirac point, gated small), rises linearly
(`numpy.polyfit` over $|\varepsilon| < 0.5t$), and peaks at the van Hove
energies $\pm t$ (the saddle at $M$; `numpy.argmax` location gated). Plot the
full $g(\varepsilon)$ — the fingerprint by which tunneling spectroscopists
recognize graphene.

In [ ]:
# (solution hidden on the public site)


### Validation 4 — the massless signatures

The fitted cone slope must be $3ta/2$ to $10^{-3}$; the density of states at
the Dirac point must be under $2\%$ of its van Hove peak; the peak must sit at
$|\varepsilon| = t$ within a bin; and the onset must be genuinely linear
(positive fitted slope with small intercept).

In [ ]:
validate.close(v_fermi, 1.5, "the Fermi velocity 3ta/2", rtol=1e-3)
validate.check(
    g_dirac < 0.02 * float(g_gr.max()),
    "the DOS vanishes at the Dirac point",
    f"g(0) = {g_dirac:.4f} vs peak {float(g_gr.max()):.3f}",
)
validate.close(
    vh_peak, 1.0, "the van Hove peak at the M-saddle energy", rtol=0.0, atol=0.03
)
validate.check(
    lin_slope > 0.0 and abs(lin_icept) < 0.05,
    "the onset is linear through zero",
    f"slope {lin_slope:.3f}, intercept {lin_icept:+.3f}",
)

## Exercise 5 — Hellmann–Feynman: bands begin to push atoms

Everything so far treated the lattice as given; real crystals choose their own
geometry, and the bridge is the force. The Hellmann–Feynman theorem (met as an
exercise tool since [§7.23](../07-quantum-statistical-mechanics/second-quantization.ipynb))
states that for an exact eigenstate, $dE/d\lambda = \langle\partial\hat
H/\partial\lambda\rangle$ — no wavefunction-response term survives. In a
tight-binding chain whose bonds alternate as $t(1 \pm \delta)$ (the Setup
helper `dimer_chain`), the parameter $\delta$ is a caricature of atomic
displacement, and the theorem's two sides are both cheaply computable.

**Part a)** For the 80-site chain at half filling (40 electrons, spin-paired
in the 40 lowest orbitals) and $\delta = 0.2$: compute $dE/d\delta$ by the
central difference of total energies at $\delta \pm 10^{-3}$
(`numpy.linalg.eigh` sums), and independently as
$2\sum_j^{\mathrm{occ}} \langle\varphi_j|\partial H/\partial\delta|\varphi_j\rangle$
(the derivative matrix has entries $(-1)^i$ on the bonds; `numpy` quadratic
forms). The two must agree at the finite-difference floor.

**Part b)** Now let the chain choose: compute the electronic energy at
$\delta = 0, 0.05, 0.1, 0.2$ and observe it *fall monotonically* — the Peierls
instability. A half-filled one-dimensional metal always profits from
dimerizing (the gap opens exactly at the Fermi points, lowering every occupied
state near them); balanced against the elastic cost $\propto\delta^2$ of real
springs, a finite dimerization wins, and the chain insulates itself. This is
why polyacetylene alternates its bonds — and the physics behind the SSH model
whose topology [§8.12](berry-wannier-ssh.ipynb) will unravel.

In [ ]:
# (solution hidden on the public site)


### Validation 5 — forces certified, instability observed

The two force routes must agree within the finite-difference floor
($10^{-4}$ relative), and the electronic energy must fall strictly
monotonically with $\delta$: the Peierls gain, gated.

In [ ]:
validate.close(force_fd, force_hf, "Hellmann-Feynman vs central differences", rtol=1e-4)
validate.check(
    bool(np.all(np.diff(E_el) < 0.0)),
    "the half-filled chain always profits from dimerizing",
    f"E falls {E_el[0]:.2f} -> {E_el[-1]:.2f}",
)

:::{admonition} With your assistant
:class: tip
The honeycomb generalizes: breaking the A/B sublattice symmetry with on-site
energies $\pm\Delta$ (the boron-nitride pattern) turns Eq.
{eq}`eq-tb-graphene`'s diagonal into $(\Delta, -\Delta)$ and gaps the cones.
Have your assistant build the $2\times2$ Bloch Hamiltonian with
$\Delta = 0.2t$ and compute the bands, then run the check that is yours
alone: the gap at $\mathbf K$ must equal exactly $2\Delta$
(`numpy.isclose`, `rtol=1e-12`), and the bands must be massive there —
quadratic, not linear (a `numpy.polyfit` of $\varepsilon(q)$ near $K$ with
the linear coefficient vanishing). Mass from broken symmetry: the check is
yours.
:::

## Notebook summary

Movement III opened with one matrix element doing a semester of work. The
chain's cosine band came from the LCAO secular equation with its honest
overlap correction (edges $-1.667t$ and $+2.5t$ at $s = 0.1$: antibonding
pushed harder, the H$_2^+$ asymmetry in a solid). Van Hove's dimensional
signatures were fitted, not recited: the chain's edge exponent $-0.48$
against $-1/2$, and the square lattice's logarithmic saddle prefactor
$-0.0504$ against the analytic $-1/2\pi^2 = -0.0507$. Graphene reduced to
one complex function whose zero at $\mathbf K$ is the cube roots of unity
summing to zero — $|f(K)| = 6\times10^{-16}$ — with the measured cone slope
$v_F = 1.49999$ against $3ta/2$ and a density of states vanishing linearly
between van Hove peaks pinned at $\pm t$. The Hellmann–Feynman theorem
certified its force against central differences at $10^{-6}$ relative, and
the Peierls computation closed the loop from bands to structure: the
half-filled chain's electronic energy falls monotonically with dimerization,
so one-dimensional metals insulate themselves — polyacetylene's story, and
the stage on which the next notebooks build.

## Outlook

- Tight binding truncates a plane-wave-exact problem to a few orbitals; the
  complementary expansion — plane waves, cutoffs, and the pseudopotential
  idea that makes them affordable — is
  [§8.10](plane-waves-pseudopotentials.ipynb), and their empirical marriage
  produces real silicon in [§8.11](epm-band-structures.ipynb).
- The dimerized chain built here *is* the SSH model;
  [§8.12](berry-wannier-ssh.ipynb) asks what its two dimerization patterns
  differ by, and the answer (a quantized Berry phase and protected edge
  states) opens the volume's window on topology.
- Hopping plus on-site repulsion is the Hubbard model of
  [§8.13](hubbard-model.ipynb) — tight binding meeting Movement I's
  correlation problem head-on.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()